# Adapted from https://milvus.io/docs/multi-vector-search.md

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('multi-qa-mpnet-base-cos-v1')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [2]:
from docling.document_converter import DocumentConverter
from docling.chunking import HybridChunker

converter = DocumentConverter()
chunker = HybridChunker()

In [3]:
import pickle

with open("docs.pickle", "rb") as f:
    docs = pickle.load(f)

In [4]:
for subjects in docs['https://services.rt.nyu.edu/docs/hpc/getting_started/intro/']:
    print(subjects)

https://services.rt.nyu.edu/docs/hpc/getting_started/intro/


In [5]:
from tqdm import tqdm

# Do all docling conversion + embedding BEFORE any MilvusClient exists.
# milvus-lite 3.x runs its gRPC server as in-process threads; forking
# (which docling's model backends can trigger) while that server is
# alive corrupts the process and crashes the kernel. Deferring the
# client/insert step until after this loop avoids that entirely.
records = []

for guide in tqdm(docs.keys()):
    for subject in docs[guide]:
        DOC_SOURCE = subject
        try:
            doc = converter.convert(source=DOC_SOURCE).document
            texts = [chunk.text for chunk in chunker.chunk(doc)]

            for text in texts:
                records.append({"text": text, "dense": model.encode(text)})
        except Exception as e:
            print(f"Skipping {DOC_SOURCE}: {e}")

100%|███████████████████████████████████████████████████████████████████████████████| 121/121 [00:49<00:00,  2.46it/s]


In [6]:
%%capture
from pymilvus import MilvusClient, DataType, Function, FunctionType, AnnSearchRequest, RRFRanker

client = MilvusClient("./milvus_demo.db")

In [7]:
schema = client.create_schema()

schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=2048, enable_analyzer=True)
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=768)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048, 'enable_analyzer': True}}, {'name': 'sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>}, {'name': 'dense', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'enable_dynamic_field': False, 'enable_namespace': False}

## Build the Milvus schema and collection (after embedding is done)

In [8]:
bm25_function = Function(
    name="text_bm25_emb", # Function name
    input_field_names=["text"], # Name of the VARCHAR field containing raw text data
    output_field_names=["sparse"], # Name of the SPARSE_FLOAT_VECTOR field reserved to store generated embeddings
    function_type=FunctionType.BM25, # Set to `BM25`
)

schema.add_function(bm25_function)

{'auto_id': True, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': True}, {'name': 'text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048, 'enable_analyzer': True}}, {'name': 'sparse', 'description': '', 'type': <DataType.SPARSE_FLOAT_VECTOR: 104>, 'is_function_output': True}, {'name': 'dense', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 768}}], 'enable_dynamic_field': False, 'enable_namespace': False, 'functions': [{'name': 'text_bm25_emb', 'description': '', 'type': <FunctionType.BM25: 1>, 'input_field_names': ['text'], 'output_field_names': ['sparse'], 'params': {}}]}

In [9]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="sparse",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={
        "inverted_index_algo": "DAAT_MAXSCORE",
        "bm25_k1": 1.2,
        "bm25_b": 0.75
    }
)

index_params.add_index(
    field_name="dense",
    index_name="text_dense_index",
    index_type="AUTOINDEX",
    metric_type="IP"
)

In [10]:
# Drop existing collection with the same name if it exists
if client.has_collection("nyu_rt_docs"):
    client.drop_collection("nyu_rt_docs")

client.create_collection(
    collection_name='nyu_rt_docs',
    schema=schema,
    index_params=index_params
)

In [11]:
BATCH_SIZE = 50

for i in tqdm(range(0, len(records), BATCH_SIZE)):
    client.insert('nyu_rt_docs', records[i:i + BATCH_SIZE])

100%|████████████████████████████████████████████████████████████████████████████████| 27/27 [00:00<00:00, 103.22it/s]


In [12]:
query = "How do I request an HPC account?"

search_params_sparse = {
    "data": [query],
    "anns_field": "sparse",
    "param": {"drop_ratio_search": 0.2},
    "limit": 5,

}
sparse_request = AnnSearchRequest(**search_params_sparse)

search_params_dense = {
    "data": [model.encode(query)],
    "anns_field": "dense",
    "param": {"nprobe": 10},
    "limit": 5
}
dense_request = AnnSearchRequest(**search_params_dense)

In [13]:
reqs = [sparse_request, dense_request]
ranker = RRFRanker()

res = client.hybrid_search(
    collection_name="nyu_rt_docs",
    reqs=reqs,
    ranker=ranker,
    limit=3,
    output_fields=["text"],  # Return id and species
)
for hits in res:
    print("Hybrid Search results:")
    for hit in hits:
        print("-----------------------------------------")
        print(f"{hit.entity.get('text')}")

Hybrid Search results:
-----------------------------------------
To request an NYU HPC account please log in to
[NYU Identity Management service](https://identity.it.nyu.edu/)
and follow the link to "Request HPC account" by clicking on the hamburger icon on the top left of the page and selecting "Manage Access" . We have a walkthrough of how to
[request an account through IIQ](/docs/hpc/getting_started/HPC_Accounts/walkthrough_request_hpc_account)
. If you are a student, alumni or an external collaborator you will need an NYU faculty sponsor.
-----------------------------------------
[Previous How to Approve an HPC Account Request](/docs/hpc/getting_started/HPC_Accounts/walkthrough_approve_hpc_account_request)
[Next HPC Accounts for Sponsored External Collaborators](/docs/hpc/getting_started/HPC_Accounts/hpc_accounts_external_collaborators)
-----------------------------------------
If your HPC Account is due for renewal, you will get an update on your dashboard that will suggest that y